# 11.4 Modifying models

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Several commands are provided to help you make limited changes to the current
`model`, without modifying the `model` file or issuing a full `reset`. This section describes
commands that completely remove or redefine `model` components, and that temporarily
`drop` constraints, `fix` variables, or relax integrality restrictions on variables.
Removing or redefining `model` components
The `delete` command removes a previously declared `model` component, provided
that no other components use it in their declarations. The form of the command is simply
`delete` followed by a comma-separated list of names of `model` components:

```ampl
ampl: model dietobj.mod;
ampl: data dietobj.dat;
ampl: delete Total_Number, Diet_Min;
```

Normally you cannot `delete` a set, parameter, or variable, because it is declared for use
later in the `model`; but you can `delete` any `objective` or constraint. You can also specify a
component "name" of the form `check` n to `delete` the nth `check` statement in the current
`model`.

The `purge` command has the same form, but with the keyword `purge` in place of
`delete`. It removes not only the listed components, but also all components that depend
on them either directly (by referring to them) or indirectly (by referring to their dependents).
Thus for example in diet.mod we have

```ampl
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];
var Buy {j in FOOD} >= f_min[j], <= f_max[j];
minimize Total_Cost: sum {j in FOOD} cost[j] * Buy[j];
```

The command `purge` f_min deletes parameter f_min and the components whose declarations
refer to f_min, including parameter f_max and variable Buy. It also deletes
`objective` Total_Cost, which depends indirectly on f_min through its reference to
Buy.

If you're not sure which components depend on some given component, you can use
the `xref` command to find out:

```ampl
ampl: xref f_min;
# 4 entities depend on f_min:
f_max
Buy
Total_Cost
Diet
```

Like `delete` and `purge`, the `xref` command can be applied to any list of `model` components.

Once a component has been removed by `delete` or `purge`, any previously hidden
meaning of the component's name becomes visible again. After a constraint named
prod is deleted, for instance, AMPL again recognizes prod as an iterated multiplication
operator ([Table 7-1](../07/7_2_arithmetic_expressions.ipynb#`table`-7-1)).

If there is no previously hidden meaning, the name of a component removed by
`delete` or `purge` becomes again unused, and may subsequently be declared as the
name of any new component of any type. If you only want to make some relatively limited
modifications to a declaration, however, then you will probably find `redeclare` to
be more convenient. You can change any component's declaration by writing the keyword
`redeclare` followed by the complete revised declaration that you would like to
substitute. Looking again at diet.mod, for example,

```ampl
ampl: redeclare param f_min {FOOD} > 0 integer;
```

changes only the validity conditions on f_min. The declarations of all components that
depend on f_min are left unchanged, as are any values previously read for f_min.

A list of all component types to which `delete`, `purge`, `xref`, and `redeclare`
may be applied is given in A.18.5.

Changing the `model`: `fix`, `unfix`; `drop`, `restore`
The simplest (but most drastic) way to change the `model` is by issuing the command
`reset`, which expunges all of the current `model` and `data`. Following `reset`, you can
issue new `model` and `data` commands to set up a different optimization problem; the
effect is like typing quit and then restarting AMPL, except that options are not `reset` to
their default values. If your operating system or your graphical environment for AMPL
allows you to edit files while keeping AMPL active, `reset` is valuable for debugging and
experimentation; you may make changes to the `model` or `data` files, type `reset`, then
read in the modified files. (If you need to escape from AMPL to run a text editor, you can
use the shell command described in Section A.21.1.)
The `drop` command instructs AMPL to ignore certain constraints or objectives of the
current `model`. As an example, the constraints of [Figure 5-1](../05/5_3_set_operations.ipynb#fig-5-1) initially include

```ampl
subject to Diet_Max {i in MAXREQ}:
       sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];
```

A `drop` command can specify a particular one of these constraints to ignore:

```ampl
drop Diet_Max["CAL"];
```

or it may specify all constraints or objectives indexed by some set:

```ampl
drop {i in MAXNOT} Diet_Max[i];
```

where MAXNOT has previously been defined as some subset of MAXREQ. The entire collection
of constraints can be ignored by

```ampl
drop {i in MAXREQ} Diet_Max[i];
```

or more simply:

```ampl
drop Diet_Max;
```

In general, this command consists of the keyword `drop`, an optional indexing expression,
and a constraint name that may be subscripted. Successive `drop` commands have a
cumulative effect.

The `restore` command reverses the effect of `drop`. It has the same syntax, except
for the keyword `restore`.
The `fix` command fixes specified variables at their current values, as if there were a
constraint that the variables must equal these values; the `unfix` command reverses the
effect. These commands have the same syntax as `drop` and `restore`, except that they
name variables rather than constraints. For example, here we initialize all variables of
our diet problem to their lower bounds, `fix` all variables representing foods that have more
than 1200 mg of sodium per package, and optimize over the remaining variables:

```ampl
ampl: let {j in FOOD} Buy[j] := f_min[j];
ampl: fix {j in FOOD: amt["NA",j] > 1200} Buy[j];
ampl: solve;
MINOS 5.5: optimal solution found.
7 iterations, objective 86.92
Objective = Total_Cost['A&P']
ampl: display {j in FOOD} (Buy[j].lb,Buy[j],amt["NA",j]);
:    Buy[j].lb    Buy[j] amt['NA',j] :=
BEEF      2       2            938
CHK       2       2           2180
FISH      2      10            945
HAM       2       2            278
MCH       2       9.42857     1182
MTL       2      10            896
SPG       2       2           1329
TUR       2       2           1397
;
```

Rather than setting and fixing the variables in separate statements, you can add an assignment
phrase to the `fix` command:

```ampl
ampl: fix {j in FOOD: amt["NA",j] > 1200} Buy[j] := f_min[j];
```

The `unfix` command works in the same way, to reverse the effect of `fix` and optionally
also `reset` the value of a variable.

Relaxing integrality
Changing option `relax_integrality` from its default of 0 to any nonzero value:

```ampl
option relax_integrality 1;
```

tells AMPL to ignore all restrictions of variables to `integer` values. Variables declared
`integer` get whatever bounds you specified for them, while variables declared
`binary` are given a lower bound of zero and an upper bound of one. To `restore` the integrality
restrictions, set the `relax_integrality` option back to 0.

A variable's name followed by the suffix .relax indicates its current integrality
relaxation status: 0 if integrality is enforced, nonzero otherwise. You can make use of
this suffix to relax integrality on selected variables only. For example,

```ampl
let Buy['CHK'].relax = 1
```

relaxes integrality only on the variable Buy['CHK'], while

```ampl
let {j in FOOD: f_min[j] > allow_frac} Buy[j].relax := 1;
```

relaxes integrality on all Buy variables for foods that have a minimum purchase of at
least some cutoff parameter allow_frac.

Some of the solvers that work with AMPL provide their own directives for relaxing
integrality, but these do not necessarily have the same effect as AMPL's
`relax_integrality` option or .relax suffix. The distinction is due to the effects
of AMPL's problem simplification, or presolve, stage (Section 14.1). AMPL drops integrality
restrictions before the presolve phase, so that the solver receives a true continuous
relaxation of the original `integer` problem. If the relaxation is performed by the solver,
however, then the integrality restrictions are still in effect during AMPL's presolve phase,
and AMPL may perform some additional tightening and simplification as a result.

As a simple example, suppose that diet `model` variable declarations are written to
allow the food limits f_max to be adjusted by setting an additional parameter, scale:

```ampl
var Buy {j in FOOD} integer >= f_min[j], <= scale * f_max[j];
```

In our example of [Figure 2-3](../02/2_4_generalizations_to_blending_economics_and_scheduling.ipynb#fig-2-3), all of the f_max values are 10; suppose that also we set
scale to 0.95. First, here are the results of solving the unrelaxed problem:

```ampl
ampl: option relax_integrality;
option relax_integrality 0;
ampl: let scale := 0.95;
ampl: solve;
CPLEX 8.0.0: optimal integer solution; objective 122.89
6 MIP simplex iterations
0 branch-and-bound nodes
```

When no relaxation is specified in AMPL, presolve sees that all the variables have upper
limits of 9.5, and since it knows that the variables must take `integer` values, it rounds
these limits down to 9. Then these limits are sent to the solver, where they remain even if
we specify a solver directive for integrality relaxation:

```ampl
ampl: option cplex_options 'relax';
ampl: solve;
CPLEX 8.0.0: relax
Ignoring integrality of 8 variables.
CPLEX 8.0.0: optimal solution; objective 120.2421057
2 dual simplex iterations (0 in phase I)
ampl: display Buy;
Buy [*] :=
BEEF 8.39898
  CHK 2
FISH 2
  HAM 9
  MCH 9
  MTL 9
  SPG 8.93436
  TUR 2
;
```

If instead option `relax_integrality` is set to 1, presolve leaves the upper limits at 9.5
and sends those to the solver, with the result being a less constrained problem and

hence a lower `objective` value:

```ampl
ampl: option relax_integrality 1;
ampl: solve;
CPLEX 8.0.0: optimal solution; objective 119.1507545
3 dual simplex iterations (0 in phase I)
ampl: display Buy;
Buy [*] :=
BEEF 6.8798
  CHK 2
FISH 2
  HAM 9.5
  MCH 9.5
  MTL 9.5
  SPG 9.1202
  TUR 2
;
```

Variables that were at upper bound 9 in the previous solution are now at upper bound 9.5.

The same situation can arise in much less obvious circumstances, and can lead to
unexpected results. In general, the optimal value of an `integer` program under AMPL's
`relax_integrality` option may be lower (for minimization) or higher (for maximization)
than the optimal value reported by the solver's relaxation directive.